Original Event Log
↓
Single Block Filtering
↓
RMG_receive
RMG_delivery
RMG_mixed
↓
Resource = Block
↓
Petri Net Discovery
↓
Parameter Discovery
↓
Save Everything

OUTPUT:
1. baseline_log.csv

2. baseline_model.pnml

3. baseline_parameters.pkl


In [2]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import prosit

import pm4py
import pm4py.objects.log.importer.xes.importer as xes_importer
from prosit import SimulatorParameters, SimulatorEngine
from datetime import datetime
from pm4py.objects.log.obj import EventLog
import pandas as pd
import pm4py

from prosit.simulator import (
    SimulatorParameters,
    SimulatorEngine
)

In [8]:
# =====================================================
# BASELINE LOG
# =====================================================

log_df = pd.read_csv(
    "../../data/processed/CTB/s4_eventlog_scenario_block_resource_log_v2.csv"
)

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    log_df[col] = pd.to_datetime(
        log_df[col]
    )

log_df.head()

,case:concept:name,concept:name,org:resource,enabled:timestamp,start:timestamp,time:timestamp,process_flow_type,n_containers,n_stops,n_deliveries,...,gate_demand,rmg_demand,vc_demand,mt_demand,gate_utilization,rmg_utilization,vc_utilization,mt_utilization,block,block_utilization
0,TP_14848272603,Gate In,Res.GateIn,2026-03-02 07:03:00,2026-03-02 07:06:00,2026-03-02 07:06:00,delivery,1,1,1,...,193.0,139.6,19.3,26.5,0.64,0.63,0.53,0.82,T22,0.7600
1,TP_14848272603,RMG_delivery,T22,2026-03-02 07:10:00,2026-03-02 07:13:00,2026-03-02 07:15:00,delivery,1,1,1,...,193.0,139.6,19.3,26.5,0.64,0.63,0.53,0.82,T22,0.7600
2,TP_14848272603,Gate Out,Res.GateOut,2026-03-02 07:20:00,2026-03-02 07:18:00,2026-03-02 07:18:00,delivery,1,1,1,...,193.0,139.6,19.3,26.5,0.64,0.63,0.53,0.82,T22,0.7600
3,TP_14848282603,Gate In,Res.GateIn,2026-03-02 06:58:00,2026-03-02 07:06:00,2026-03-02 07:06:00,receive,1,1,0,...,193.0,139.6,19.3,26.5,0.64,0.63,0.53,0.82,T18,0.7622
4,TP_14848282603,RMG_receive,T18,2026-03-02 07:10:00,2026-03-02 07:12:00,2026-03-02 07:18:00,receive,1,1,0,...,193.0,139.6,19.3,26.5,0.64,0.63,0.53,0.82,T18,0.7622


In [9]:
# =====================================================
# PM4PY LOG
# =====================================================

event_log = pm4py.format_dataframe(
    log_df,
    case_id="case:concept:name",
    activity_key="concept:name",
    timestamp_key="time:timestamp"
)

net, im, fm = pm4py.discover_petri_net_inductive(
    event_log
)

In [10]:
from pm4py.objects.conversion.log import converter as log_converter
event_log_df = pm4py.format_dataframe(
    log_df,
    case_id="case:concept:name",
    activity_key="concept:name",
    timestamp_key="time:timestamp"
)

event_log = log_converter.apply(
    event_log_df
)

print(type(event_log))

<class 'pm4py.objects.log.obj.EventLog'>


In [11]:
# =====================================================
# PARAMETER DISCOVERY
# =====================================================

params = SimulatorParameters(
    net,
    im,
    fm
)

print(params.resources)

params.discover_from_eventlog(
    event_log,
    max_depth_tree=5
)
print(params.resources)
params.to_json(
    "baseline_parameters.json"
)

['auto']
Resources discovery...
Data attributes discovery...
PATCHED 1.0 script is running
Feature discovery...


100%|██████████| 67391/67391 [02:30<00:00, 449.10it/s]


Transition Probabilities discovery...


transition models: 100%|██████████| 88/88 [02:35<00:00,  1.76s/it]


Resource Weights discovery...


...

Calendars discovery...
Execution Time discovery...


exec-time models: 100%|██████████| 29/29 [00:13<00:00,  2.11it/s]


Waiting Time discovery...


waiting-time models: 100%|██████████| 26/26 [00:27<00:00,  1.06s/it]


Arrival Time discovery...
['Res.GateIn', 'Res.GateOut', 'T13', 'T14', 'T24', 'T17', 'T16', 'T12', 'T21', 'T20', 'T11', 'T23', 'T22', 'T27', 'T06', 'T10', 'T09', 'T18', 'T15', 'T07', 'T08', 'T19', 'T26', 'T25', 'LL', 'HO2']


In [12]:
import pm4py

# Event Log laden
event_log = pm4py.convert_to_event_log(
    pm4py.format_dataframe(
        log_df,
        case_id="case:concept:name",
        activity_key="concept:name",
        timestamp_key="time:timestamp"
    )
)

# Petri Net entdecken
net, im, fm = pm4py.discover_petri_net_inductive(
    event_log
)

# speichern
pm4py.write_pnml(
    net,
    im,
    fm,
    "baseline_model.pnml"
)

In [13]:
import pickle

with open(
    "baseline_parameters.pkl",
    "wb"
) as f:

    pickle.dump(
        params,
        f
    )

In [14]:
# =====================================================
# BASELINE
# =====================================================

baseline_engine = SimulatorEngine(
    params
)

baseline_log = baseline_engine.apply(
    n_traces=20000
)

baseline_log.to_csv(
    "baseline_simulation.csv",
    index=False
)

Simulating Cases: 100%|██████████| 20000/20000 [00:32<00:00, 608.85it/s]
